In [1]:
from PyPDF2 import PdfReader
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
import evaluate, accelerate
import numpy as np
from transformers import Trainer, TrainingArguments
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling
import os

c:\Users\pr\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:



pdf_path = r"C:\Python 2k25\4_LLMs_Basics\Day_5_Mini_Project_Custom_LLM_Chatbot\notes\general_QnAs.pdf"

reader = PdfReader(pdf_path)

full_text = ""                              # Create an empty box to store ALL text from ALL pages
for page in reader.pages:                   # Go through the PDF page by page.
    page_text = page.extract_text()
    if page_text:                           # If text exists:-> add it to full_text: then -> add a newline so pages don’t glue together
        full_text += page_text + "\n"

import re               # re mean regex: Why regex? Because: answers span multiple lines + page breaks exist + spacing is inconsistent. So, Simple .split() cannot handle this safely.
import json

pattern = re.compile(
    r"prompt:\s*(.*?)\s*response:\s*(.*?)(?=\s*prompt:|\Z)",
    re.DOTALL | re.IGNORECASE )                     # allow matching across new lines
                                                    # Prompt / prompt / PROMPT all work

matches = pattern.findall(full_text)                # This returns : Each item as: (prompt_text, response_text)

qa_pairs = []
seen = set()  # to avoid duplicates

for prompt, response in matches:
    prompt = prompt.strip()
    response = response.strip()

    key = (prompt.lower(), response.lower())        # Create a duplicate-check fingerprint. Lowercase ensures: Same content ≠ counted twice due to casing.
    if key in seen:
        continue                                     # ← exits loop early (append never happens) if duplicate comes
    seen.add(key)                                    # ← remember this pair (in other case)

    qa_pairs.append({                                # ← runs ONLY if not skipped
        "prompt": prompt,
        "response": response
    })

print(f"Extracted {len(qa_pairs)} QnA pairs")




Extracted 20 QnA pairs


In [3]:
with open("extracted_qna.json", "w", encoding="utf-8") as f:
    json.dump(qa_pairs, f, indent=2, ensure_ascii=False)

print("Saved clean QnA dataset to extracted_qna.json")


Saved clean QnA dataset to extracted_qna.json


In [4]:
dataset = load_dataset("json", data_files = "extracted_qna.json")
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'response'],
        num_rows: 20
    })
})


In [5]:
repo_id="openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(repo_id)

tokenizer.pad_token = tokenizer.eos_token                 # uses end-of-sequence as pad

model = AutoModelForCausalLM.from_pretrained(repo_id)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

c:\Users\pr\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pr\.cache\huggingface\hub\models--openai-community--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
def preprocess (dataset):
    full_text = [p + '\n' + r for p, r in zip(dataset['prompt'], dataset['response'])]
    return tokenizer(full_text, truncation=True, padding='max_length', max_length=128)

mapped_dataset = dataset.map(preprocess, batched=True)
print(mapped_dataset)
print(mapped_dataset['train'].features)

data_train = mapped_dataset['train']

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt', 'response', 'input_ids', 'attention_mask'],
        num_rows: 20
    })
})
{'prompt': Value('string'), 'response': Value('string'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}


In [7]:
config = LoraConfig(r=8, lora_alpha=16, target_modules=["c_attn"], task_type="CAUSAL_LM")
peft_model = get_peft_model(model = model, peft_config=config)

c:\Users\pr\AppData\Local\Programs\Python\Python311\Lib\site-packages\peft\tuners\lora\layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [8]:
# Data collator for CausalLM (shifts labels, ignores -100 on prompt)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Not masked LM
)

In [9]:
accuracy = evaluate.load('accuracy')

def compute_metrics_function (eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions = predictions, references = labels)

In [10]:
arguments = TrainingArguments(
    output_dir = r"C:\Python 2k25\4_LLMs_Basics\Day_5_Mini_Project_Custom_LLM_Chatbot",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    eval_strategy='no',  # Skip eval for speed
    save_strategy='epoch'
)
trainer =Trainer(
    model = peft_model,
    args = arguments,
    train_dataset= data_train,
    data_collator=data_collator,
    eval_dataset= None, # not needed here
    compute_metrics = compute_metrics_function
)

In [11]:
history = trainer.train()
trainer.save_model("Model_gpt2_fine_tuned_for_custom_ChatBOT")
tokenizer.save_pretrained("tokenizers_gpt2_fine_tuned_for_custom_ChatBOT")

c:\Users\pr\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


c:\Users\pr\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\pr\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


('tokenizers_gpt2_fine_tuned_for_custom_ChatBOT\\tokenizer_config.json',
 'tokenizers_gpt2_fine_tuned_for_custom_ChatBOT\\special_tokens_map.json',
 'tokenizers_gpt2_fine_tuned_for_custom_ChatBOT\\vocab.json',
 'tokenizers_gpt2_fine_tuned_for_custom_ChatBOT\\merges.txt',
 'tokenizers_gpt2_fine_tuned_for_custom_ChatBOT\\added_tokens.json',
 'tokenizers_gpt2_fine_tuned_for_custom_ChatBOT\\tokenizer.json')